In [33]:
import qim3d
from pathlib import Path
from typing import Literal
from comptest.image_transform import DataTransform, DtypeDataTransform
import numpy as np

In [34]:
RECONSTRUCTION_INPUTS_DIR = Path(
    "/dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/naturarv3d_compression_study/reconstruction_inputs"
).resolve()
assert RECONSTRUCTION_INPUTS_DIR.exists()

In [35]:
# DATASET: str = "Canislupus_BM_2600"
# DATASET: str = "Canislupus_CZM_712"
DATASET: str = "Canislupus_UAM_18197"


In [36]:
PROJECTION_FILENAME = DATASET + ".xtekct"
PROJECTIONS_FILE = RECONSTRUCTION_INPUTS_DIR / DATASET / PROJECTION_FILENAME
assert PROJECTIONS_FILE.exists(), f"Projections file {PROJECTIONS_FILE} does not exist"

RECONSTRUCTION_FOLDER = RECONSTRUCTION_INPUTS_DIR / DATASET / (DATASET + "_rec")
assert RECONSTRUCTION_FOLDER.exists(), (
    f"Reconstruction folder {RECONSTRUCTION_FOLDER} does not exist"
)

RECONSTRUCTION_FILE = RECONSTRUCTION_FOLDER / (DATASET + ".vgi")
assert RECONSTRUCTION_FILE.exists(), (
    f"Reconstruction file {RECONSTRUCTION_FILE} does not exist"
)

RECONSTRUCTION_TRANSFORMED_FOLDER = Path(
    "/dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/naturarv3d_compression_study/reconstructions_transformed"
)
assert RECONSTRUCTION_TRANSFORMED_FOLDER.exists(), (
    f"Reconstruction transformed folder {RECONSTRUCTION_TRANSFORMED_FOLDER} does not exist"
)

In [37]:
vol, metadata = qim3d.io.load(RECONSTRUCTION_FILE, return_metadata=True)

In [38]:
normalizer = DataTransform.infer(vol, target_range=(0.0, 1.0))
vol_normalized = normalizer.apply(vol)
# vol_normalized =

In [39]:
# Save normalized float volume, note JPEG-XL and some other codecs need the data to be in [0, 1], thus we normalize

output_dir = RECONSTRUCTION_TRANSFORMED_FOLDER / "f32_2000" / DATASET
output_dir.mkdir(parents=True, exist_ok=True)

qim3d.io.save(
    output_dir,
    vol_normalized,
    overwrite=True,
    basename="slice_z",
    sliced_dim=0,
)

In [40]:
# Turn into uint16
uint16_normalizer = DtypeDataTransform.infer(vol, target_dtype=np.uint16)
vol_uint16 = uint16_normalizer.apply(vol)

In [41]:
vol_uint16.min(), vol_uint16.max()

(np.uint16(0), np.uint16(65535))

In [42]:
output_dir = RECONSTRUCTION_TRANSFORMED_FOLDER / "u16_2000" / DATASET
output_dir.mkdir(parents=True, exist_ok=True)

qim3d.io.save(
    output_dir,
    vol_uint16,
    basename="slice_z",
    sliced_dim=0,
)

In [43]:
def bin2(vol: np.ndarray, slab: int = 100) -> np.ndarray:
    """2x2x2 mean binning, processed in z-slabs to bound memory."""
    z, y, x = (s // 2 * 2 for s in vol.shape)  # crop odd edges
    out = np.empty((z // 2, y // 2, x // 2), dtype=np.float32)
    for z0 in range(0, z, 2 * slab):
        z1 = min(z0 + 2 * slab, z)
        block = np.asarray(vol[z0:z1, :y, :x], dtype=np.float32)
        out[z0 // 2 : z1 // 2] = block.reshape(
            (z1 - z0) // 2, 2, y // 2, 2, x // 2, 2
        ).mean(axis=(1, 3, 5))
    return out


In [44]:
vol_normalized_1k = bin2(vol_normalized)  # (1000, 1000, 1000)


In [45]:
output_dir = RECONSTRUCTION_TRANSFORMED_FOLDER / "f32_1000" / DATASET
output_dir.mkdir(parents=True, exist_ok=True)

qim3d.io.save(
    output_dir,
    vol_normalized_1k,
    basename="slice_z",
    sliced_dim=0,
)

In [46]:
uint16_1k_normalizer = DtypeDataTransform.infer(
    vol_normalized_1k, target_dtype=np.uint16
)
vol_uint16_1k = uint16_1k_normalizer.apply(vol_normalized_1k)

In [47]:
output_dir = RECONSTRUCTION_TRANSFORMED_FOLDER / "u16_1000" / DATASET
output_dir.mkdir(parents=True, exist_ok=True)

qim3d.io.save(
    output_dir,
    vol_uint16_1k,
    basename="slice_z",
    sliced_dim=0,
)